# Pesa Pulse - Customer Directory API Analysis

**Author:** Junior Data Analyst  
**Client:** Pesa Pulse (Nairobi Fintech Start-Up)  

---

## Project Overview
Pesa Pulse has recently integrated with a new regional partner who shares customer records via an API. Before launching regional marketing campaigns, the marketing team needs clear answers to three essential questions:
1. **How many customers are there?**
2. **Where are the customers located?**
3. **What does the data structure look like, and what marketing opportunities exist?**

This notebook implements the complete exploratory data analysis workflow:
- Safely retrieving customer JSON data using `requests` with robust error and timeout handling.
- Recursively exploring nested dictionary and list structures.
- Extracting and flattening key customer attributes.
- Aggregating and counting distributions using pure Python data structures (sets, loops, and dict comprehensions).
- Validating results with `pandas` DataFrames and exporting clean CSV deliverables.
- Visualizing customer distributions with presentation-ready charts.

## Task 1: Retrieve Customer Data with Requests
We implement `get_customers(url)` to fetch data from `https://jsonplaceholder.typicode.com/users` with a 10-second timeout. Any network failure is caught cleanly and prints an informative error message without exposing raw tracebacks.

In [1]:
from typing import Any, Dict, List, Set
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt

def get_customers(url: str, timeout: int = 10) -> List[Dict[str, Any]]:
    """
    Fetch customer records from the REST API endpoint.
    Handles network timeouts and HTTP errors cleanly.
    """
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Network error fetching customer data from {url}: {e}")
        return []

api_url = "https://jsonplaceholder.typicode.com/users"
raw_customers = get_customers(api_url)
print(f"Successfully retrieved {len(raw_customers)} customer records.")

## Task 2: Explore Nested Dictionaries and Lists
We implement a recursive `explore(record, indent=0)` function to traverse the structure of a customer record, displaying every key and its corresponding Python data type with hierarchical indentation.

In [2]:
def explore(record: Any, indent: int = 0) -> None:
    """
    Recursively walks a nested structure (dict or list) and prints each key with its Python type.
    """
    spacing = " " * indent
    if isinstance(record, dict):
        for key, value in record.items():
            print(f"{spacing}{key}: {type(value).__name__}")
            if isinstance(value, (dict, list)):
                explore(value, indent + 4)
    elif isinstance(record, list):
        for index, item in enumerate(record):
            print(f"{spacing}[{index}]: {type(item).__name__}")
            if isinstance(item, (dict, list)):
                explore(item, indent + 4)

# Explore the first customer record
print("Nested structure of customer record #1:")
print("-" * 45)
explore(raw_customers[0])
print("-" * 45)

## Task 3: Extract Customer Attributes
We implement `extract_attributes(customers)` to produce a flat dictionary for each customer. The extracted fields include `customer_id`, `name`, `username`, `email`, `email_domain` (TLD), `phone`, `city`, `zipcode`, `latitude` (float), `longitude` (float), `company` (name), and `website`.

In [3]:
def extract_attributes(customers: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Extracts and flattens nested customer records.
    Converts latitude and longitude into floats and isolates the top-level domain (email_domain).
    """
    flattened = []
    for c in customers:
        address = c.get("address", {})
        geo = address.get("geo", {})
        company = c.get("company", {})
        email = c.get("email", "")
        tld = email.split(".")[-1] if "." in email else ""
        
        row = {
            "customer_id": int(c.get("id")),
            "name": c.get("name", ""),
            "username": c.get("username", ""),
            "email": email,
            "email_domain": tld,
            "phone": c.get("phone", ""),
            "city": address.get("city", ""),
            "zipcode": address.get("zipcode", ""),
            "latitude": float(geo.get("lat", 0.0)),
            "longitude": float(geo.get("lng", 0.0)),
            "company": company.get("name", ""),
            "website": c.get("website", ""),
        }
        flattened.append(row)
    return flattened

customers_flat = extract_attributes(raw_customers)
print(f"Sample flattened customer record:")
customers_flat[0]

## Task 4: Use Sets to Find Unique Cities
We implement `unique_cities(customers)` using a set comprehension to find all distinct cities where customers reside.

In [4]:
def unique_cities(customers: List[Dict[str, Any]]) -> Set[str]:
    """
    Find all distinct cities represented in the customer data using a set comprehension.
    """
    return {
        c.get("city") or (c.get("address", {}).get("city") if isinstance(c.get("address"), dict) else "")
        for c in customers
    }

cities = unique_cities(customers_flat)
num_customers = len(customers_flat)
num_cities = len(cities)

print(f"Number of customers: {num_customers}")
print(f"Number of unique cities: {num_cities}")
print(f"\nMarketing Insight:")
print("Because there are exactly 10 unique cities across 10 customers (a 1:1 ratio), customer residency is entirely fragmented across the map with zero geographic clustering, indicating that local billboard or single-city marketing events are premature and Pesa Pulse should focus on broad digital campaigns instead.")

## Task 5: Count Customers by City and Email Domain
We implement:
- `city_counts(customers)` using plain loops and dictionaries (without `collections.Counter` or `pandas`).
- `email_domain_counts(customers)` using a dictionary comprehension.
Both results are sorted from most to least common.

In [5]:
def city_counts(customers: List[Dict[str, Any]]) -> Dict[str, int]:
    """
    Count customers per city using standard loop and dict accumulation.
    Sorted by count descending, then city ascending.
    """
    counts: Dict[str, int] = {}
    for c in customers:
        city = c.get("city") or (c.get("address", {}).get("city") if isinstance(c.get("address"), dict) else "")
        if city:
            counts[city] = counts.get(city, 0) + 1
    return dict(sorted(counts.items(), key=lambda item: (-item[1], item[0])))

def email_domain_counts(customers: List[Dict[str, Any]]) -> Dict[str, int]:
    """
    Count customers per email domain (TLD) using a dictionary comprehension.
    Sorted by count descending, then domain ascending.
    """
    domains = [
        c.get("email_domain") if "email_domain" in c else c.get("email", "").split(".")[-1]
        for c in customers
    ]
    counts = {d: domains.count(d) for d in domains if d}
    return dict(sorted(counts.items(), key=lambda item: (-item[1], item[0])))

c_counts = city_counts(customers_flat)
d_counts = email_domain_counts(customers_flat)

print("City counts (sorted descending):")
for city, cnt in c_counts.items():
    print(f"  {city:20s}: {cnt}")

print("\nEmail domain counts (sorted descending):")
for dom, cnt in d_counts.items():
    print(f"  .{dom:19s}: {cnt}")

## Task 6: Convert Data into a DataFrame
We load the flattened records into a `pandas.DataFrame` and cross-verify with `value_counts()` that pandas agrees with our manual counts.

In [6]:
def to_dataframe(rows: List[Dict[str, Any]]) -> pd.DataFrame:
    """Convert list of flat dictionaries into a pandas DataFrame."""
    return pd.DataFrame(rows)

df = to_dataframe(customers_flat)
print(f"DataFrame Shape: {df.shape}")
print("\nData Types:")
print(df.dtypes)

# Confirm agreement
pandas_city_counts = df["city"].value_counts().to_dict()
assert pandas_city_counts == c_counts, "Mismatch between pandas and manual counts!"
print("\nVerification Successful: pandas value_counts() matches manual counts perfectly.")
df.head()

## Task 7: Export Results to CSV
We write `output/customers.csv` and `output/city_counts.csv` without the pandas index, following the column naming in `expected_output.md`.

In [7]:
def export_results(df: pd.DataFrame, city_counts_dict: Dict[str, int], output_dir: str = "output") -> None:
    """Export cleaned customer records and city counts to CSV without index."""
    os.makedirs(output_dir, exist_ok=True)
    
    # Export customers.csv
    cust_path = os.path.join(output_dir, "customers.csv")
    cols = [
        "customer_id", "name", "username", "email", "email_domain",
        "phone", "city", "zipcode", "latitude", "longitude",
        "company", "website"
    ]
    df[cols].to_csv(cust_path, index=False)
    print(f"Saved customers table to: {cust_path}")
    
    # Export city_counts.csv
    city_path = os.path.join(output_dir, "city_counts.csv")
    city_df = pd.DataFrame([{"city": k, "customer_count": v} for k, v in city_counts_dict.items()])
    city_df = city_df.sort_values(by=["customer_count", "city"], ascending=[False, True])
    city_df.to_csv(city_path, index=False)
    print(f"Saved city counts table to: {city_path}")

export_results(df, c_counts, "output")

## Task 8: Visualize the Customer Distribution
We plot:
1. `output/customer_distribution.png`: Bar chart of customers by city.
2. `output/email_domain_distribution.png`: Bar chart of customers by email domain (TLD).

### Chart Comparison & Evaluation
- **City Distribution Chart:** Since every customer lives in a different city, every bar has a frequency of 1. While this accurately proves that customers are evenly dispersed, it provides low differentiation for marketing prioritization.
- **Email Domain (TLD) Chart:** The email domain chart reveals meaningful grouping — `.biz` accounts for 30% (3 of 10) of the customer base, followed by single occurrences across other domains. This offers actionable intelligence for marketing, showing a high concentration of commercial/business entities that can be targeted with tailored B2B fintech messaging.

In [8]:
def plot_distribution(
    counts: Dict[str, int],
    path: str,
    title: str = "Customer Distribution by City",
    xlabel: str = "City",
    ylabel: str = "Customer Count"
) -> None:
    """Save a formatted bar chart of distributions with data labels."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    categories = list(counts.keys())
    values = list(counts.values())
    
    plt.figure(figsize=(10, 5.5))
    bars = plt.bar(categories, values, color="#1f77b4", edgecolor="#0e4b75", width=0.6)
    
    plt.title(title, fontsize=13, fontweight="bold", pad=12)
    plt.xlabel(xlabel, fontsize=11, labelpad=8)
    plt.ylabel(ylabel, fontsize=11, labelpad=8)
    plt.xticks(rotation=45, ha="right", fontsize=10)
    plt.yticks(range(0, max(values) + 2), fontsize=10)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    
    for bar in bars:
        h = bar.get_height()
        plt.annotate(
            f"{int(h)}",
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )
    
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.show()
    print(f"Saved chart to: {path}")

plot_distribution(c_counts, "output/customer_distribution.png", "Customer Distribution by City (Pesa Pulse)", "City", "Customer Count")
plot_distribution(d_counts, "output/email_domain_distribution.png", "Customer Distribution by Email Domain / TLD (Pesa Pulse)", "Email Domain (TLD)", "Customer Count")

## Final Console Findings
Matching the exact specification in `expected_output.md`:

In [9]:
most_common_city = list(c_counts.keys())[0]
most_common_count = list(c_counts.values())[0]

print("## Console findings")
print(f"Customers retrieved : {len(customers_flat)}")
print(f"Unique cities       : {len(cities)}")
print(f"Most common city    : {most_common_city} ({most_common_count})")
print(f"Email domains       : {d_counts}")
print(f"Chart saved to      : output/customer_distribution.png")